# Module 2: Tools & ReAct Agents

**Day 4 — LangGraph Agents, Memory, HITL & MCP**

## What you will learn
- **@tool decorator**: turn any Python function into an LLM-callable tool
- **ToolNode**: LangGraph's built-in node that executes tool calls
- **tools_condition**: built-in conditional edge for the ReAct loop
- **bind_tools**: attach tool definitions to an LLM
- **ReAct agent loop**: Think -> Act -> Observe -> Think ...
- **MockLLMWithTools**: test agents without API keys

## ReAct Loop
```
User Query -> Agent (LLM decides tool) -> ToolNode (executes) -> Agent (sees result) -> Final Answer
```


In [ ]:
import sys
sys.path.insert(0, '../src')
print('Path configured.')

## 1. The @tool Decorator

The `@tool` decorator converts a Python function into a LangChain tool.
The LLM sees the function's name, description, and parameter schema — and decides when to call it.

In [ ]:
from day4.tools_agents import calculator, get_weather, search_web

# Inspect tool metadata
for t in [calculator, get_weather, search_web]:
    print(f'Tool: {t.name}')
    print(f'Description: {t.description[:80]}')
    print()

In [ ]:
# Call tools directly
print('calculator(2**10):', calculator.invoke({'expression': '2 ** 10'}))
print('calculator(sqrt(256)):', calculator.invoke({'expression': 'sqrt(256)'}))
print('get_weather(mumbai):', get_weather.invoke({'city': 'mumbai'}))
print('search_web(langgraph):', search_web.invoke({'query': 'langgraph'}))

## 2. Building a ReAct Agent

The ReAct pattern (Yao et al., 2022) alternates between:
- **Re**asoning: LLM decides what tool to call
- **Act**ing: ToolNode executes the tool
- Observing: LLM sees the result and decides next action

LangGraph implements this as a loop: `agent -> tools -> agent -> tools -> ... -> END`

In [ ]:
from day4.tools_agents import build_tool_agent, MockLLMWithTools, extract_final_answer, count_tool_calls
from langchain_core.messages import HumanMessage

# Mock LLM: always calls calculator first, then gives final answer
mock_llm = MockLLMWithTools(
    tool_name='calculator',
    tool_args={'expression': '18 * 9900'},
    final_answer='The total annual cost is Rs 1,78,200 (18 months x Rs 9,900 per month).'
)

agent = build_tool_agent(mock_llm, [calculator, get_weather, search_web])

result = agent.invoke({'messages': [HumanMessage('What is 18 months x Rs 9900 per month?')]})

print(f'Messages in conversation: {len(result["messages"])}')
print(f'Tool calls made: {count_tool_calls(result)}')
print(f'Final answer: {extract_final_answer(result)}')

In [ ]:
# Trace the full conversation
print('Full conversation trace:')
print('=' * 60)
for i, msg in enumerate(result['messages']):
    msg_type = type(msg).__name__
    content = msg.content[:80] if msg.content else ''
    tool_calls = getattr(msg, 'tool_calls', [])
    
    print(f'{i+1}. [{msg_type}]')
    if content:
        print(f'   Content: {content}')
    if tool_calls:
        print(f'   Tool calls: {[tc["name"] for tc in tool_calls]}')

## 3. Building Your Own Tool

Any Python function decorated with `@tool` becomes callable by the LLM.
Industry examples: database query tools, API wrappers, file system tools.

In [ ]:
from langchain_core.tools import tool

@tool
def stock_price(ticker: str) -> str:
    """Get the current stock price for an Indian company (NSE/BSE).
    
    Args:
        ticker: Stock ticker symbol (e.g., TCS, INFY, WIPRO).
    
    Returns:
        Current price and change as string.
    """
    # Mock data — in production: use NSE/BSE API
    prices = {
        'TCS':   'TCS: Rs 3,842.50 (+1.2%)',
        'INFY':  'INFY: Rs 1,456.75 (-0.3%)',
        'WIPRO': 'WIPRO: Rs 478.20 (+0.8%)',
        'HCL':   'HCL Technologies: Rs 1,234.60 (+0.5%)',
    }
    return prices.get(ticker.upper(), f'Ticker {ticker} not found')

print('Testing custom tool:')
print(stock_price.invoke({'ticker': 'TCS'}))
print(stock_price.invoke({'ticker': 'INFY'}))
print(f'Tool name: {stock_price.name}')
print(f'Tool schema: {stock_price.args_schema.model_json_schema()}')

## 4. Real Agent with OpenAI (requires API key)

Once you have an API key, replace the mock with a real LLM:

In [ ]:
# Real agent pattern (requires OPENAI_API_KEY)
# Uncomment when you have an API key:
#
# import os
# from dotenv import load_dotenv
# from langchain_openai import ChatOpenAI
#
# load_dotenv('../.env')
# llm = ChatOpenAI(model='gpt-4o-mini', temperature=0)
# agent = build_tool_agent(llm, [calculator, get_weather, search_web, stock_price])
# result = agent.invoke({'messages': [HumanMessage('What is the weather in Mumbai and the TCS stock price?')]})
# print(extract_final_answer(result))

print('Real agent requires OPENAI_API_KEY.')
print('See .env.example in the project root.')

## Databricks Bridge

In [ ]:
# Databricks integration: tools can call Databricks SQL, run jobs, etc.
#
# from databricks import sql
# from langchain_core.tools import tool
#
# @tool
# def query_databricks(sql_query: str) -> str:
#     """Run a SQL query on Databricks Unity Catalog."""
#     with sql.connect(server_hostname=..., http_path=..., access_token=...) as conn:
#         with conn.cursor() as cursor:
#             cursor.execute(sql_query)
#             rows = cursor.fetchall()
#             return str(rows[:10])  # return first 10 rows
#
# agent = build_tool_agent(llm, [query_databricks, get_weather, calculator])

print('Databricks SQL tool pattern: any Python function can be a LangGraph tool.')